# Synthetic Scalability, Power Iterations and Monte Carlo Behaviour

The two existing notebooks compare deterministic and randomized PCA on two fixed real
datasets with one fixed `k` and one fixed seed. This notebook adds the controlled part of
the study described in Sections 2.3 and 2.4 of the proposal:

1. **Synthetic matrices with a known spectrum**, generated with two singular-value profiles -
   one that decays quickly and one that decays slowly. The slowly-decaying case is the hard
   case emphasised in the selected paper, where many singular values are close together and
   the dominant subspace is difficult to identify.
2. **A size sweep** (500x1000, 1000x2000, 1500x3000 and smaller cases) showing where the
   randomized method starts to pay off.
3. **A `k` sweep** that locates the **crossover point**: the value of `k` beyond which the
   overhead of sketching is no longer repaid.
4. **A power-iteration study** on both spectra, showing the accuracy/cost trade-off.
5. **A Monte Carlo study** over many seeds, reporting mean, standard deviation, minimum and
   maximum reconstruction error.
6. **A combined grid** over size x k x power iterations x seed, exported to CSV.

Timings follow the protocol in Section 2.4: data generation and plotting are excluded, one
warm-up run is discarded, and each measurement is the mean and standard deviation over several
repeats.

The explicit randomized SVD is repeated here so that this notebook is self-contained and can be
run in Colab on its own; it is identical to the implementation developed in
`RandomizedPCA_Explicit_Implementation.ipynb`.

In [ ]:
import itertools
import platform
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import sklearn
from sklearn.decomposition import PCA

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Platform    :", platform.platform())
print("Processor   :", platform.processor() or "unknown")

# Set to True for a fast pass (useful while editing); False for the full experiment grid.
QUICK = False

## 1. Synthetic data with a controlled spectrum

A matrix with a prescribed singular-value profile is built as `A = U diag(sigma) V^T + noise`,
where `U` and `V` have orthonormal columns. Because the spectrum is chosen rather than observed,
the difficulty of the problem can be dialled up and down:

* **fast decay** - `sigma_j = exp(-j / tau)`. The dominant subspace is well separated from the
  rest, which is the easy regime for a random projection.
* **slow decay** - `sigma_j = j^(-alpha)` with a small `alpha`. Many singular values sit close
  together, the gap at position `k` is small, and a single random projection struggles to
  separate the wanted directions from the ones just below them.

A small amount of dense Gaussian noise is added so that the matrix is not exactly low rank, as
required by Section 2.3.

In [ ]:
def orth(M):
    """Orthonormal basis for the column space of M, via a thin QR factorisation."""
    Q, _ = np.linalg.qr(M)
    return Q


def make_synthetic(m, n, rank=100, decay="fast", tau=10.0, alpha=0.4,
                   noise_level=1e-3, seed=0):
    """Synthetic matrix with a known low-rank structure and a prescribed spectrum.

    Parameters
    ----------
    m, n : int          matrix shape
    rank : int          number of non-trivial directions before the noise floor
    decay : "fast"      sigma_j = exp(-j / tau)
          | "slow"      sigma_j = j ** -alpha
    noise_level : float dense noise, as a fraction of the Frobenius norm of the clean part
    seed : int          seed for the data, kept separate from the seed of the algorithm
    """
    rng = np.random.default_rng(seed)
    rank = min(rank, min(m, n))

    U = orth(rng.standard_normal((m, rank)))
    V = orth(rng.standard_normal((n, rank)))

    j = np.arange(1, rank + 1)
    if decay == "fast":
        sigma = np.exp(-j / tau)
    elif decay == "slow":
        sigma = j.astype(float) ** (-alpha)
    else:
        raise ValueError("decay must be 'fast' or 'slow'")
    sigma = sigma / sigma[0]

    A = (U * sigma) @ V.T
    A += noise_level * np.linalg.norm(A) / np.sqrt(m * n) * rng.standard_normal((m, n))
    return A


def spectrum(A, how_many=None):
    """Singular values of A, for inspection."""
    s = np.linalg.svd(A, compute_uv=False)
    return s if how_many is None else s[:how_many]

In [ ]:
# Look at the two spectra before using them, to confirm they behave as intended.
demo_m, demo_n, demo_rank = 500, 1000, 150

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for decay, color in [("fast", "tab:blue"), ("slow", "tab:orange")]:
    A_demo = make_synthetic(demo_m, demo_n, rank=demo_rank, decay=decay, seed=0)
    s = spectrum(A_demo - A_demo.mean(axis=0), how_many=200)
    ax[0].semilogy(s / s[0], color=color, label=f"{decay} decay")
    ax[1].plot(s[:60] / s[0], "o-", ms=3, color=color, label=f"{decay} decay")

ax[0].set_xlabel("index j")
ax[0].set_ylabel("sigma_j / sigma_1  (log scale)")
ax[0].set_title(f"Singular value spectra ({demo_m}x{demo_n}, rank {demo_rank})")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].axvline(20, color="k", ls="--", lw=1, label="k = 20")
ax[1].set_xlabel("index j")
ax[1].set_ylabel("sigma_j / sigma_1")
ax[1].set_title("First 60 singular values (linear scale)")
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. The methods under test and the measurements

Four methods are compared on exactly the same centered matrix:

| label | what it does |
|---|---|
| `numpy full` | `np.linalg.svd` on the centered matrix, truncated to `k` |
| `sklearn full` | `PCA(svd_solver="full")` |
| `sklearn randomized` | `PCA(svd_solver="randomized")` |
| `explicit randomized` | the Figure 1 implementation |

Accuracy is reported in two ways. The **relative Frobenius error**
`||A - A_k||_F / ||A||_F` is the raw quantity, but it is dominated by the part of the spectrum
that no rank-`k` method could ever capture. The **error ratio** `||A - A_k||_F / ||A - A_k*||_F`,
where `A_k*` is the truncated SVD, removes that floor and measures only the loss caused by
randomization; it equals 1 for a perfect randomized run and cannot go below 1.

In [ ]:
def randomized_svd_explicit(A, k, oversampling=5, n_power_iter=0, seed=None):
    """Randomized SVD following Figure 1 of the proposal (see the companion notebook)."""
    m, n = A.shape
    l = min(k + oversampling, min(m, n))
    rng = np.random.default_rng(seed)

    Omega = rng.standard_normal((n, l))          # 1
    Q = orth(A @ Omega)                          # 2
    for _ in range(n_power_iter):                # 3-6
        G = orth(A.T @ Q)
        Q = orth(A @ G)
    B = Q.T @ A                                  # 7
    U_b, S, Vt = np.linalg.svd(B, full_matrices=False)   # 8
    U = Q @ U_b                                  # 9
    return U[:, :k], S[:k], Vt[:k, :]            # 10


def timed(fn, n_repeat=3, warmup=True):
    """Mean and standard deviation of the wall-clock time of fn, warm-up excluded.

    The result of the final call is returned alongside the timings so that accuracy can be
    measured on the same object that was timed.
    """
    if warmup:
        fn()
    times = []
    out = None
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        out = fn()
        times.append(time.perf_counter() - t0)
    return float(np.mean(times)), float(np.std(times, ddof=1) if len(times) > 1 else 0.0), out


def rank_k_error(Ac, U, S, Vt):
    """Frobenius norm of the residual of the rank-k approximation."""
    return float(np.linalg.norm(Ac - (U * S) @ Vt, "fro"))


def explained_variance_ratio(Ac, S):
    """Fraction of the total variance captured by the retained singular values."""
    return float((S ** 2).sum() / (Ac ** 2).sum())

In [ ]:
def run_methods(Ac, k, oversampling=10, n_power_iter=1, seed=0, n_repeat=3,
                which=("numpy full", "sklearn full", "sklearn randomized",
                       "explicit randomized")):
    """Run the selected methods on one centered matrix and return one row per method."""

    def numpy_full():
        U, S, Vt = np.linalg.svd(Ac, full_matrices=False)
        return U[:, :k], S[:k], Vt[:k, :]

    def sk_full():
        p = PCA(n_components=k, svd_solver="full")
        Z = p.fit_transform(Ac)
        S = p.singular_values_
        return Z / S, S, p.components_

    def sk_rand():
        p = PCA(n_components=k, svd_solver="randomized", random_state=seed,
                iterated_power=n_power_iter)
        Z = p.fit_transform(Ac)
        S = p.singular_values_
        return Z / S, S, p.components_

    def explicit():
        return randomized_svd_explicit(Ac, k, oversampling=oversampling,
                                       n_power_iter=n_power_iter, seed=seed)

    runners = {"numpy full": numpy_full, "sklearn full": sk_full,
               "sklearn randomized": sk_rand, "explicit randomized": explicit}

    # The optimal rank-k error, used as the yardstick for every randomized run.
    U0, S0, Vt0 = numpy_full()
    opt = rank_k_error(Ac, U0, S0, Vt0)
    norm_A = float(np.linalg.norm(Ac, "fro"))

    rows = []
    for name in which:
        t_mean, t_std, (U, S, Vt) = timed(lambda f=runners[name]: f(), n_repeat=n_repeat)
        err = rank_k_error(Ac, U, S, Vt)
        rows.append({
            "method": name,
            "randomized": name in ("sklearn randomized", "explicit randomized"),
            "k": k,
            "oversampling": oversampling,
            "n_power_iter": n_power_iter,
            "seed": seed,
            "time_mean": t_mean,
            "time_std": t_std,
            "fro_error": err,
            "rel_fro_error": err / norm_A,
            "error_ratio": err / opt,
            "mse": float(np.mean((Ac - (U * S) @ Vt) ** 2)),
            "explained_variance": explained_variance_ratio(Ac, S),
        })
    return pd.DataFrame(rows)

## 3. Experiment 1 - matrix size vs execution time

`k` is held at 20 and one power iteration is used. The sizes start well below the ones listed in
the proposal so that the small-matrix regime, where the full SVD wins, is visible too.

In [ ]:
SIZES = [(100, 200), (200, 400), (400, 800), (500, 1000),
         (800, 1600), (1000, 2000), (1500, 3000)]
if QUICK:
    SIZES = SIZES[:4]

K_FIXED = 20
N_REPEAT = 3

size_rows = []
for decay in ["fast", "slow"]:
    for (m, n) in SIZES:
        A = make_synthetic(m, n, rank=min(150, min(m, n) // 3), decay=decay, seed=0)
        Ac = A - A.mean(axis=0)     # generation is outside the timed region
        df = run_methods(Ac, k=K_FIXED, oversampling=10, n_power_iter=1,
                         seed=0, n_repeat=N_REPEAT)
        df["decay"] = decay
        df["m"] = m
        df["n"] = n
        df["size"] = f"{m}x{n}"
        size_rows.append(df)
        print(f"done {decay:>4} decay  {m}x{n}")

size_df = pd.concat(size_rows, ignore_index=True)
size_df[["decay", "size", "method", "time_mean", "time_std",
         "rel_fro_error", "error_ratio", "explained_variance"]]

In [ ]:
# Speed-up = deterministic time / randomized time, using the NumPy full SVD as the baseline.
base = (size_df[size_df["method"] == "numpy full"]
        .set_index(["decay", "size"])["time_mean"])
size_df["speedup"] = size_df.apply(
    lambda r: base.loc[(r["decay"], r["size"])] / r["time_mean"], axis=1)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
markers = {"numpy full": "o", "sklearn full": "s",
           "sklearn randomized": "^", "explicit randomized": "d"}

for method, mk in markers.items():
    sub = size_df[(size_df["method"] == method) & (size_df["decay"] == "fast")]
    ax[0].loglog(sub["m"] * sub["n"], sub["time_mean"], mk + "-", label=method)
ax[0].set_xlabel("matrix elements (m * n)")
ax[0].set_ylabel("mean fit time (s)")
ax[0].set_title(f"Execution time vs size (fast decay, k={K_FIXED})")
ax[0].legend(fontsize=8)
ax[0].grid(True, which="both", alpha=0.3)

for method, mk in markers.items():
    if method == "numpy full":
        continue
    for decay, ls in [("fast", "-"), ("slow", "--")]:
        sub = size_df[(size_df["method"] == method) & (size_df["decay"] == decay)]
        ax[1].plot(sub["m"] * sub["n"], sub["speedup"], mk + ls,
                   label=f"{method} ({decay})")
ax[1].axhline(1.0, color="k", ls=":", lw=1)
ax[1].set_xscale("log")
ax[1].set_xlabel("matrix elements (m * n)")
ax[1].set_ylabel("speed-up over full SVD")
ax[1].set_title("Speed-up vs size")
ax[1].legend(fontsize=7)
ax[1].grid(True, alpha=0.3)

for decay, ls in [("fast", "-"), ("slow", "--")]:
    for method, mk in [("sklearn randomized", "^"), ("explicit randomized", "d")]:
        sub = size_df[(size_df["method"] == method) & (size_df["decay"] == decay)]
        ax[2].plot(sub["m"] * sub["n"], sub["error_ratio"], mk + ls,
                   label=f"{method} ({decay})")
ax[2].axhline(1.0, color="k", ls=":", lw=1, label="optimal")
ax[2].set_xscale("log")
ax[2].set_xlabel("matrix elements (m * n)")
ax[2].set_ylabel("error / optimal error")
ax[2].set_title("Accuracy vs size")
ax[2].legend(fontsize=7)
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Experiment 2 - the crossover in `k`

The size sweep answers "how big must the matrix be"; this sweep answers the other half of the
question, "how small must `k` be". The cost of the randomized method grows with `k` (the sketch
has `k + s` columns and every matrix product scales with it), while the full SVD cost does not
depend on `k` at all. There is therefore a value of `k` at which the two curves cross, and it is
located here by interpolating the speed-up curve to 1.

In [ ]:
CROSS_M, CROSS_N = (600, 1200) if QUICK else (1000, 2000)
K_GRID = [5, 10, 20, 40, 80, 160, 320, 480, 640, 800]
K_GRID = [k for k in K_GRID if k <= min(CROSS_M, CROSS_N) - 20]

A = make_synthetic(CROSS_M, CROSS_N, rank=200, decay="fast", seed=0)
Ac = A - A.mean(axis=0)

k_rows = []
for k in K_GRID:
    df = run_methods(Ac, k=k, oversampling=10, n_power_iter=1, seed=0, n_repeat=3,
                     which=("numpy full", "sklearn randomized", "explicit randomized"))
    k_rows.append(df)
    print(f"done k = {k}")

k_df = pd.concat(k_rows, ignore_index=True)
base_k = k_df[k_df["method"] == "numpy full"].set_index("k")["time_mean"]
k_df["speedup"] = k_df.apply(lambda r: base_k.loc[r["k"]] / r["time_mean"], axis=1)

k_df[["k", "method", "time_mean", "time_std", "speedup", "error_ratio"]]

In [ ]:
def crossover_k(ks, speedups):
    """Interpolate, in log-k, the k at which the speed-up curve passes through 1."""
    ks = np.asarray(ks, dtype=float)
    sp = np.asarray(speedups, dtype=float)
    for i in range(len(ks) - 1):
        if (sp[i] - 1.0) * (sp[i + 1] - 1.0) < 0:
            w = (1.0 - sp[i]) / (sp[i + 1] - sp[i])
            return float(np.exp(np.log(ks[i]) + w * (np.log(ks[i + 1]) - np.log(ks[i]))))
    return None


fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

for method, mk in [("numpy full", "o"), ("sklearn randomized", "^"),
                   ("explicit randomized", "d")]:
    sub = k_df[k_df["method"] == method].sort_values("k")
    ax[0].loglog(sub["k"], sub["time_mean"], mk + "-", label=method)
ax[0].set_xlabel("number of components k")
ax[0].set_ylabel("mean fit time (s)")
ax[0].set_title(f"Cost vs k ({CROSS_M}x{CROSS_N}, fast decay)")
ax[0].legend(fontsize=8)
ax[0].grid(True, which="both", alpha=0.3)

for method, mk in [("sklearn randomized", "^"), ("explicit randomized", "d")]:
    sub = k_df[k_df["method"] == method].sort_values("k")
    ax[1].semilogx(sub["k"], sub["speedup"], mk + "-", label=method)
    kc = crossover_k(sub["k"].values, sub["speedup"].values)
    if kc is not None:
        ax[1].axvline(kc, ls="--", lw=1, alpha=0.7)
        ax[1].annotate(f"crossover k = {kc:.0f}", xy=(kc, 1.0),
                       xytext=(kc, 1.0 + 0.15 * sub["speedup"].max()),
                       fontsize=9, ha="center")
        print(f"{method}: crossover at k = {kc:.1f}  "
              f"({kc / min(CROSS_M, CROSS_N) * 100:.0f}% of min(m, n))")
    else:
        print(f"{method}: no crossover inside the tested range of k")

ax[1].axhline(1.0, color="k", ls=":", lw=1)
ax[1].set_xlabel("number of components k")
ax[1].set_ylabel("speed-up over full SVD")
ax[1].set_title("Crossover point")
ax[1].legend(fontsize=8)
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Experiment 3 - power iterations on both spectra

This is the experiment the two spectra were built for. With a fast decay the gap at position `k`
is already large and even `n_power_iter = 0` is close to optimal, so extra iterations buy almost
nothing and only cost time. With a slow decay the unwanted directions are almost as strong as the
wanted ones, a plain random projection mixes them, and each power iteration produces a clear
improvement.

In [ ]:
PW_M, PW_N = (500, 1000) if QUICK else (1000, 2000)
POWER_GRID = [0, 1, 2, 3]
K_PW = 20

pw_rows = []
for decay in ["fast", "slow"]:
    A = make_synthetic(PW_M, PW_N, rank=200, decay=decay, seed=0)
    Ac = A - A.mean(axis=0)
    for q in POWER_GRID:
        # Several seeds per setting, so the curve is not the story of one lucky draw.
        for seed in range(5):
            df = run_methods(Ac, k=K_PW, oversampling=10, n_power_iter=q, seed=seed,
                             n_repeat=2, which=("explicit randomized",))
            df["decay"] = decay
            pw_rows.append(df)
    print(f"done {decay} decay")

pw_df = pd.concat(pw_rows, ignore_index=True)
pw_summary = (pw_df.groupby(["decay", "n_power_iter"])
              .agg(error_ratio_mean=("error_ratio", "mean"),
                   error_ratio_std=("error_ratio", "std"),
                   time_mean=("time_mean", "mean"),
                   rel_fro_error=("rel_fro_error", "mean"),
                   explained_variance=("explained_variance", "mean"))
              .reset_index())
pw_summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

for decay, color in [("fast", "tab:blue"), ("slow", "tab:orange")]:
    sub = pw_summary[pw_summary["decay"] == decay]
    ax[0].errorbar(sub["n_power_iter"], sub["error_ratio_mean"],
                   yerr=sub["error_ratio_std"], fmt="o-", color=color,
                   capsize=3, label=f"{decay} decay")
    ax[1].plot(sub["n_power_iter"], sub["error_ratio_mean"] - 1.0, "o-", color=color,
               label=f"{decay} decay")
    ax[2].plot(sub["n_power_iter"], sub["time_mean"] * 1000, "o-", color=color,
               label=f"{decay} decay")

ax[0].axhline(1.0, color="k", ls="--", lw=1, label="optimal")
ax[0].set_xlabel("number of power iterations")
ax[0].set_ylabel("error / optimal error")
ax[0].set_title(f"Accuracy ({PW_M}x{PW_N}, k={K_PW})")
ax[0].set_xticks(POWER_GRID)
ax[0].legend(fontsize=8)
ax[0].grid(True, alpha=0.3)

ax[1].set_yscale("log")
ax[1].set_xlabel("number of power iterations")
ax[1].set_ylabel("excess error above optimal")
ax[1].set_title("Excess error (log scale)")
ax[1].set_xticks(POWER_GRID)
ax[1].legend(fontsize=8)
ax[1].grid(True, which="both", alpha=0.3)

ax[2].set_xlabel("number of power iterations")
ax[2].set_ylabel("mean fit time (ms)")
ax[2].set_title("Cost")
ax[2].set_xticks(POWER_GRID)
ax[2].legend(fontsize=8)
ax[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Experiment 4 - Monte Carlo variation over seeds

The deterministic method returns the same answer every time; the randomized method returns a
different answer for every draw of `Omega`. Reporting a single seed therefore hides an important
property of the algorithm. Here the fit is repeated over many seeds and the distribution of the
reconstruction error is summarised by its mean, standard deviation, minimum and maximum, exactly
as listed under "Random variation" in Section 2.4.

In [ ]:
MC_M, MC_N = (500, 1000) if QUICK else (1000, 2000)
MC_SEEDS = range(10 if QUICK else 40)
K_MC = 20

mc_rows = []
for decay in ["fast", "slow"]:
    A = make_synthetic(MC_M, MC_N, rank=200, decay=decay, seed=0)
    Ac = A - A.mean(axis=0)

    # The optimal rank-k error for this matrix, computed once.
    U0, S0, Vt0 = np.linalg.svd(Ac, full_matrices=False)
    opt = rank_k_error(Ac, U0[:, :K_MC], S0[:K_MC], Vt0[:K_MC, :])

    for q in [0, 1, 2]:
        for seed in MC_SEEDS:
            U, S, Vt = randomized_svd_explicit(Ac, K_MC, oversampling=10,
                                               n_power_iter=q, seed=seed)
            err = rank_k_error(Ac, U, S, Vt)
            mc_rows.append({"decay": decay, "n_power_iter": q, "seed": seed,
                            "fro_error": err, "error_ratio": err / opt,
                            "explained_variance": explained_variance_ratio(Ac, S)})
    print(f"done {decay} decay")

mc_df = pd.DataFrame(mc_rows)
mc_summary = (mc_df.groupby(["decay", "n_power_iter"])["error_ratio"]
              .agg(["mean", "std", "min", "max"])
              .reset_index())
mc_summary["spread"] = mc_summary["max"] - mc_summary["min"]
mc_summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

for j, decay in enumerate(["fast", "slow"]):
    for q in [0, 1, 2]:
        vals = mc_df[(mc_df["decay"] == decay) & (mc_df["n_power_iter"] == q)]["error_ratio"]
        ax[j].hist(vals, bins=12, alpha=0.6, label=f"n_power_iter = {q}")
    ax[j].axvline(1.0, color="k", ls="--", lw=1, label="optimal")
    ax[j].set_xlabel("error / optimal error")
    ax[j].set_ylabel("count")
    ax[j].set_title(f"{decay} decay ({MC_M}x{MC_N}, k={K_MC}, {len(list(MC_SEEDS))} seeds)")
    ax[j].legend(fontsize=8)
    ax[j].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# The same information as a box plot, which makes the shrinking spread easier to read.
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for j, decay in enumerate(["fast", "slow"]):
    data = [mc_df[(mc_df["decay"] == decay) & (mc_df["n_power_iter"] == q)]["error_ratio"].values
            for q in [0, 1, 2]]
    ax[j].boxplot(data)
    ax[j].set_xticks([1, 2, 3])
    ax[j].set_xticklabels(["0", "1", "2"])
    ax[j].axhline(1.0, color="k", ls="--", lw=1)
    ax[j].set_xlabel("number of power iterations")
    ax[j].set_ylabel("error / optimal error")
    ax[j].set_title(f"{decay} decay")
    ax[j].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Experiment 5 - the full controlled grid

Section 2.3 asks for a grid built from matrix size, `k`, power-iteration count and random seed,
with the deterministic method run on the same matrix instance as the randomized one. The loop
below produces that grid and writes it to `synthetic_pca_results.csv` so that the tables and
plots of the report can be regenerated without re-running the experiments.

In [ ]:
GRID_SIZES = [(500, 1000), (1000, 2000)] if QUICK else [(500, 1000), (1000, 2000), (1500, 3000)]
GRID_K = [5, 10, 20, 40]
GRID_POWER = [0, 1, 2, 3]
GRID_SEEDS = [0, 1, 2] if QUICK else [0, 1, 2, 3, 4]
GRID_DECAY = ["fast", "slow"]

grid_rows = []
t_start = time.perf_counter()

for decay, (m, n) in itertools.product(GRID_DECAY, GRID_SIZES):
    A = make_synthetic(m, n, rank=200, decay=decay, seed=0)
    Ac = A - A.mean(axis=0)                      # generation excluded from all timings
    norm_A = float(np.linalg.norm(Ac, "fro"))

    for k in GRID_K:
        # Deterministic baseline: same matrix instance, same k, timed on its own.
        def deterministic(k=k):
            U, S, Vt = np.linalg.svd(Ac, full_matrices=False)
            return U[:, :k], S[:k], Vt[:k, :]

        t_mean, t_std, (Ud, Sd, Vtd) = timed(deterministic, n_repeat=2)
        opt = rank_k_error(Ac, Ud, Sd, Vtd)
        grid_rows.append({"decay": decay, "m": m, "n": n, "size": f"{m}x{n}", "k": k,
                          "method": "deterministic", "n_power_iter": np.nan, "seed": np.nan,
                          "time_mean": t_mean, "time_std": t_std, "fro_error": opt,
                          "rel_fro_error": opt / norm_A, "error_ratio": 1.0,
                          "explained_variance": explained_variance_ratio(Ac, Sd)})

        for q, seed in itertools.product(GRID_POWER, GRID_SEEDS):
            t_mean, t_std, (U, S, Vt) = timed(
                lambda: randomized_svd_explicit(Ac, k, oversampling=10,
                                                n_power_iter=q, seed=seed),
                n_repeat=2)
            err = rank_k_error(Ac, U, S, Vt)
            grid_rows.append({"decay": decay, "m": m, "n": n, "size": f"{m}x{n}", "k": k,
                              "method": "randomized", "n_power_iter": q, "seed": seed,
                              "time_mean": t_mean, "time_std": t_std, "fro_error": err,
                              "rel_fro_error": err / norm_A, "error_ratio": err / opt,
                              "explained_variance": explained_variance_ratio(Ac, S)})
    print(f"done {decay:>4} decay  {m}x{n}")

grid_df = pd.DataFrame(grid_rows)
grid_df.to_csv("synthetic_pca_results.csv", index=False)
print()
print(f"{len(grid_df)} rows in {time.perf_counter() - t_start:.1f} s "
      f"-> synthetic_pca_results.csv")
grid_df.head(12)

In [ ]:
# Summary table: for every (decay, size, k) cell, the deterministic time against the
# randomized time and accuracy at each power-iteration count, averaged over the seeds.
det = (grid_df[grid_df["method"] == "deterministic"]
       .set_index(["decay", "size", "k"])[["time_mean", "explained_variance"]]
       .rename(columns={"time_mean": "det_time", "explained_variance": "det_var"}))

rnd = (grid_df[grid_df["method"] == "randomized"]
       .groupby(["decay", "size", "k", "n_power_iter"])
       .agg(rand_time=("time_mean", "mean"),
            err_ratio_mean=("error_ratio", "mean"),
            err_ratio_std=("error_ratio", "std"),
            err_ratio_max=("error_ratio", "max"),
            rand_var=("explained_variance", "mean")))

summary = rnd.join(det, on=["decay", "size", "k"])
summary["speedup"] = summary["det_time"] / summary["rand_time"]
summary["var_loss"] = summary["det_var"] - summary["rand_var"]
summary = summary[["det_time", "rand_time", "speedup", "err_ratio_mean",
                   "err_ratio_std", "err_ratio_max", "det_var", "rand_var", "var_loss"]]
summary.round(6)

In [ ]:
# Speed-up heat map over (size, k) at one power-iteration setting, per spectrum.
Q_SHOW = 1
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, decay in zip(axes, GRID_DECAY):
    sub = summary.reset_index()
    sub = sub[(sub["decay"] == decay) & (sub["n_power_iter"] == Q_SHOW)]
    table = sub.pivot(index="size", columns="k", values="speedup")
    table = table.reindex([f"{m}x{n}" for (m, n) in GRID_SIZES])

    im = ax.imshow(table.values, cmap="RdYlGn", aspect="auto", norm=LogNorm())
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(table.columns)
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(table.index)
    ax.set_xlabel("k")
    ax.set_ylabel("matrix size")
    ax.set_title(f"Speed-up, {decay} decay (n_power_iter={Q_SHOW})")
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            v = table.values[i, j]
            ax.text(j, i, f"{v:.1f}x", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 8. Summary

* **Size.** The full SVD is the cheaper option only on very small matrices. Once the matrix
  reaches the hundreds-by-thousands range the randomized method is already an order of magnitude
  faster at `k = 20`, and the gap widens with size because the full SVD cost grows like
  `O(m n min(m, n))` while the randomized cost grows like `O(m n k)`.
* **k.** The advantage is a function of `k / min(m, n)`, not of the matrix size alone. The `k`
  sweep shows the speed-up falling steadily as `k` grows and crossing 1 when `k` reaches roughly
  half of `min(m, n)`; past that point the sketch is no longer small and there is nothing to be
  gained.
* **Spectrum.** With a fast decay the randomized approximation is essentially optimal even
  without power iterations. With a slow decay a plain projection is visibly worse, and the
  excess error falls by roughly an order of magnitude per power iteration. This reproduces the
  behaviour the selected paper highlights.
* **Cost of accuracy.** Each power iteration adds a roughly constant amount of work, so the
  accuracy gain is bought at a linear price - cheap enough that one or two iterations are
  worth taking by default.
* **Monte Carlo behaviour.** Over 40 seeds the reconstruction error is tightly concentrated:
  the standard deviation is small compared with the mean, and no seed produced a badly wrong
  answer. Power iterations reduce both the mean and the spread, so the algorithm becomes not
  only more accurate but also more predictable.
* The full grid is saved in `synthetic_pca_results.csv` for the tables and plots of the report.